# Section 7. Embedding과 VectorStore로 의미 검색하기

공개 실습용 notebook입니다. 필수 셀은 네트워크와 API 없이 실행됩니다.
`ConceptEmbedding`은 재현 가능한 수업용 모형이며 실제 언어 embedding model이 아닙니다.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore


def find_data() -> Path:
    candidates = [
        Path("../data/section07_retrieval_corpus.json"),
        Path(__file__).resolve().parents[1] / "data" / "section07_retrieval_corpus.json"
        if "__file__" in globals()
        else Path("__missing__"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("section07_retrieval_corpus.json을 찾지 못했습니다.")


records = json.loads(find_data().read_text(encoding="utf-8"))
documents = [
    Document(
        page_content=row["text"],
        metadata={
            "source_id": row["source_id"],
            "chunk_id": f"{row['source_id']}:000",
            "title": row["title"],
            "category": row["category"],
        },
    )
    for row in records
]
assert len({doc.metadata["source_id"] for doc in documents}) == len(documents)

## Cosine similarity를 작은 벡터로 확인하기

In [ ]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    if len(a) != len(b):
        raise ValueError("vector dimensions differ")
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        raise ValueError("cosine similarity is undefined for a zero vector")
    return dot / (norm_a * norm_b)


assert cosine_similarity([1.0, 0.0], [0.8, 0.1]) > cosine_similarity(
    [1.0, 0.0], [0.1, 0.9]
)
print("cosine dry run: PASS")

## 결정론적 수업용 embedding

동의어를 같은 대표 개념으로 바꾸고 안정적인 hash bucket에 세는 단순 모형입니다. 실제 embedding의
일반화 성능을 재현하지 않으며, API 없이 VectorStore의 데이터 흐름과 평가법을 연습하기 위한 장치입니다.

In [ ]:
ALIASES = {
    "비밀": "credential", "인증정보": "credential", "키": "credential", "key": "credential",
    "token": "credential", "유출": "exposure", "노출": "exposure", "공개": "exposure",
    "오류": "error", "예외": "error", "에러": "error", "질문": "report",
    "공유": "report", "검색": "retrieval", "근거": "evidence", "인용": "evidence",
    "추측": "unanswerable", "답이": "unanswerable", "환경": "environment", "설치": "environment",
    "python": "python", "uv": "python", "kernel": "python", "평가": "evaluation",
    "순위": "evaluation", "결과": "evaluation", "기록": "evaluation",
}


class ConceptEmbedding(Embeddings):
    """Offline teaching model; not a substitute for a production embedding model."""

    def __init__(self, dimensions: int = 512):
        self.dimensions = dimensions

    def _embed(self, text: str) -> list[float]:
        vector = [0.0] * self.dimensions
        normalized = text.lower()
        for surface, concept in ALIASES.items():
            normalized = normalized.replace(surface, f" {concept} ")
        tokens = re.findall(r"[0-9A-Za-z가-힣_]+", normalized)
        for token in tokens:
            concept = ALIASES.get(token, token)
            digest = hashlib.sha256(concept.encode("utf-8")).digest()
            bucket = int.from_bytes(digest[:4], "big") % self.dimensions
            vector[bucket] += 1.0
        norm = math.sqrt(sum(value * value for value in vector)) or 1.0
        return [value / norm for value in vector]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)


embedding = ConceptEmbedding()
store = InMemoryVectorStore(embedding=embedding)
ids = store.add_documents(documents)
assert len(ids) == len(documents)
print("offline index documents:", len(ids))

## lexical baseline과 vector 검색 비교

In [ ]:
def lexical_rank(query: str, docs: list[Document], k: int = 3) -> list[tuple[Document, float]]:
    query_terms = set(re.findall(r"[0-9A-Za-z가-힣_]+", query.lower()))
    ranked = []
    for doc in docs:
        terms = set(re.findall(r"[0-9A-Za-z가-힣_]+", doc.page_content.lower()))
        ranked.append((doc, len(query_terms & terms) / max(len(query_terms), 1)))
    return sorted(ranked, key=lambda item: (-item[1], item[0].metadata["source_id"]))[:k]


QUESTIONS = [
    ("direct", "API key를 어디에 공개하면 안 되나요?"),
    ("paraphrase", "비밀 인증정보가 유출된 것 같으면 어떻게 대응하나요?"),
    ("unanswerable", "연구실 정기 회의는 매주 몇 시인가요?"),
]

for kind, question in QUESTIONS:
    lexical = lexical_rank(question, documents)
    semantic = store.similarity_search_with_score(question, k=3)
    print(f"\n[{kind}] {question}")
    print(" lexical:", [(d.metadata["source_id"], round(s, 3)) for d, s in lexical])
    print(" vector :", [(d.metadata["source_id"], round(s, 3)) for d, s in semantic])
    assert all("chunk_id" in doc.metadata for doc, _ in semantic)

paraphrase_top = store.similarity_search(
    "비밀 인증정보가 유출된 것 같으면 어떻게 대응하나요?", k=1
)[0]
assert paraphrase_top.metadata["source_id"] == "security-guide"
print("\nmetadata preservation and intended paraphrase case: PASS")
print("offline fixture gate: PASS (real API quality is not verified)")

## 선택: 작고 명시적인 실제 embedding readiness gate

기본 실행은 여기서 멈추며 비용이 발생하지 않습니다. 아래 gate는 `RUN_LIVE_API=1`이고
`OPENAI_API_KEY`가 환경에 있을 때만 실행됩니다. 문서 5개를 한 번의 batch로 index하고, 고정된 질문
3개만 각각 query합니다. key의 값은 읽어 출력하지 않습니다.

In [ ]:
LIVE_MODEL = "text-embedding-3-small"
MAX_LIVE_DOCUMENTS = 5
MAX_LIVE_QUERIES = 3
MAX_LIVE_TOTAL_CHARS = 12_000


def response_token_usage(response: object) -> dict[str, int] | None:
    usage = getattr(response, "usage", None)
    if usage is None:
        return None
    fields = ("prompt_tokens", "total_tokens")
    result = {name: int(getattr(usage, name)) for name in fields if getattr(usage, name, None) is not None}
    return result or None


def live_embedding_readiness_gate() -> None:
    # Environment variable의 존재만 확인합니다. 값은 출력·저장하지 않습니다.
    if os.getenv("RUN_LIVE_API") != "1":
        print("real API gate: SKIPPED (set RUN_LIVE_API=1 to opt in)")
        return
    if not os.getenv("OPENAI_API_KEY"):
        print("real API gate: SKIPPED (OPENAI_API_KEY is not available in the environment)")
        return

    from openai import OpenAI

    document_texts = [doc.page_content for doc in documents]
    queries = [question for _, question in QUESTIONS]
    assert len(document_texts) <= MAX_LIVE_DOCUMENTS
    assert len(queries) <= MAX_LIVE_QUERIES
    total_chars = sum(map(len, document_texts + queries))
    assert total_chars <= MAX_LIVE_TOTAL_CHARS

    # 한국어/영어 혼합 입력의 정확한 token 수는 tokenizer로 별도 계산해야 합니다. 여기서는 호출 전에
    # 입력 규모를 확인하기 위한 보수적 문자 기반 범위만 제시하고, 호출 뒤 API usage를 우선 기록합니다.
    approximate_token_range = (math.ceil(total_chars / 4), total_chars)
    print("\n[real API embedding readiness plan]")
    print(" model:", LIVE_MODEL)
    print(" document inputs:", len(document_texts), f"(hard limit {MAX_LIVE_DOCUMENTS})")
    print(" query inputs:", len(queries), f"(hard limit {MAX_LIVE_QUERIES})")
    print(" total characters:", total_chars, f"(hard limit {MAX_LIVE_TOTAL_CHARS})")
    print(" approximate token range:", approximate_token_range, "(character-based; not billing exact)")
    print(" planned requests: 4 (documents once + 3 queries)")
    print(" pricing reference: $0.02 / 1M input tokens, OpenAI pricing, checked 2026-07-14; recheck before use")

    client = OpenAI()
    document_response = client.embeddings.create(model=LIVE_MODEL, input=document_texts)
    document_vectors = [item.embedding for item in document_response.data]
    usages = [response_token_usage(document_response)]
    assert len(document_vectors) == len(documents)

    top_sources: dict[str, str] = {}
    for (kind, question) in QUESTIONS:
        query_response = client.embeddings.create(model=LIVE_MODEL, input=question)
        usages.append(response_token_usage(query_response))
        query_vector = query_response.data[0].embedding
        scores = [cosine_similarity(query_vector, vector) for vector in document_vectors]
        best_index = max(range(len(scores)), key=scores.__getitem__)
        top_sources[kind] = documents[best_index].metadata["source_id"]
        print(f" {kind} top-1:", top_sources[kind], f"score={scores[best_index]:.3f}")

    known_usage = [usage for usage in usages if usage is not None]
    if known_usage:
        print(" API-reported usage by request:", known_usage)
        prompt_total = sum(item.get("prompt_tokens", item.get("total_tokens", 0)) for item in known_usage)
        print(" API-reported input token total (when exposed):", prompt_total)
    else:
        print(" API-reported usage: unavailable in these responses; use provider usage records for billing")

    # direct/paraphrase는 자체 corpus에서 기대 근거를 찾는지 검사합니다. unanswerable의 top-1은 관찰만 하며,
    # 결과가 반환됐다고 정답이 corpus에 있다는 뜻은 아닙니다.
    assert top_sources["direct"] == "security-guide"
    assert top_sources["paraphrase"] == "security-guide"
    print("real API gate: PASS (direct/paraphrase retrieval; unanswerable remains a refusal-policy case)")


live_embedding_readiness_gate()

## 정리 질문

1. lexical과 vector top-3가 달라진 질문은 무엇인가요?
2. 답이 없는 질문에서도 결과가 반환되는 이유는 무엇인가요?
3. 높은 similarity가 correctness를 보장하지 않는 사례를 하나 적으세요.